# One-step retrosynthesis evaluation with vLLM

Self-contained vLLM port of the generation loop in `code_v1.ipynb`. Nothing here imports from
or depends on that notebook -- run this file top to bottom on its own.

Same evaluation protocol as `code_v1.ipynb` and `DeepRetro/notebooks/prod_challenge.ipynb`:
canonicalize predicted precursors and ground-truth reactants with RDKit, then require equal
cardinality plus set membership. `all_correct` means every predicted precursor is in the
ground truth with matching counts; `any_correct` means at least one is.

A third metric, `maxfrag_correct`, is the MaxFrag ("classical retro-synthesis") accuracy of
Tetko et al., *Nat. Commun.* 11, 5575 (2020): only the largest predicted precursor (most heavy
atoms) has to match the largest ground-truth reactant. It ignores the smaller reagents and the
precursor count, so it scores whether the model found the principal disconnection.

Each scoring run writes, next to the per-molecule `results_vllm.csv`, the headline numbers as
`summary_vllm.json` and the printed report verbatim as `summary_vllm.txt`, and appends one row
to `results/summary_all_vllm.csv`, a running log of every scoring run across models.

## Why vLLM

The HuggingFace `model.generate()` loop in `code_v1.ipynb` runs at **~8.1 s/molecule**
(measured: 250 rows in 33.9 min). Three things cost that time, none of them the model:

1. `BATCH_SIZE = 4`, sized for the 2xT4 Kaggle box the original comments describe. This
   machine is a single RTX PRO 6000 Blackwell (96GB, sm_120); a 7B model in bf16 is ~15GB, so
   the card sits almost entirely idle.
2. **Ragged batches.** Generated lengths run mean 578 / p90 754 / max 1024 tokens, and
   `generate()` runs every batch until its *longest* member finishes -- about 1.77x (max/mean)
   of the decode steps are spent on sequences that already stopped.
3. **No prefix reuse.** `SYS_PROMPT_OPENAI` + `USER_PROMPT_OPENAI` is ~600 tokens and is
   byte-identical across all 250 rows apart from the embedded SMILES, yet it is recomputed
   from scratch on every call.

vLLM fixes all three: continuous batching, a paged KV cache, and automatic prefix caching.
Measured result: **1.12 min for 250 molecules (0.27 s/mol), a 30.1x speedup**, with top-1
scores, parse-failure rate and mean proposal count all matching the HF engine.

## Kernel

This notebook needs the **Python (vLLM)** kernel, not the one `code_v1.ipynb` uses. vLLM pins
`torch==2.13` and `transformers>=5.5`, which cannot coexist with `/venv/main` (torch 2.10,
transformers 4.57), so it lives in its own venv:

```bash
python3 -m venv /venv/vllm
/venv/vllm/bin/pip install vllm==0.29.0 rdkit pandas accelerate ipywidgets ipykernel
/venv/vllm/bin/python -m ipykernel install --user --name vllm --display-name "Python (vLLM)"
```

Verify you are on it with `import sys, vllm; print(sys.executable, vllm.__version__)` ->
`/venv/vllm/bin/python 0.29.0`.

In [2]:
from huggingface_hub import login
login()

In [8]:
import ast
import json
import os
import re
import sys
import time

import pandas as pd
import torch
from rdkit import Chem, RDLogger
from transformers import AutoTokenizer

RDLogger.DisableLog("rdApp.*")  # silence the parse errors we already handle

REPO = "/root/workspace/DFS/DeepRetro"
sys.path.insert(0, REPO)

# DeepRetro's own prompts. SYS/USER_PROMPT_OPENAI are the non-CoT pair the repo uses for
# models that do not emit <cot> tags -- the right family for instruct models like
# Olmo-3-Instruct. src/variables.py is pure string constants, so this pulls in no
# litellm/langfuse.
from src.variables import SYS_PROMPT_OPENAI, USER_PROMPT_OPENAI

# --- the only knobs that have to change to evaluate a different model ---
MODEL_ID = "zai-org/GLM-4.7-Flash"

# Qwen3's chat template defaults to thinking ON, which emits a <think>...</think> block
# before the answer. At 1024 new tokens that block eats the whole budget and the JSON never
# arrives (the smoke test below returned 502 that way). Hard switch it here; the value is
# passed to apply_chat_template in build_prompt(). Non-Qwen templates ignore the kwarg.
#
# The 2507-Thinking checkpoints are thinking-only: their template has no enable_thinking
# switch (the kwarg is silently ignored) and it opens the assistant turn with "<think>\n"
# itself, so the model's output holds only the closing </think>. Keep this True for them --
# it still selects the sampling params, token budget and the "_think" output directory.
ENABLE_THINKING = False

DATA_CSV = f"{REPO}/data/uspto_50k_test_250.csv"
# Thinking needs far more room than the answer itself; 1024 matches the HF loop otherwise.
# 4096 was too tight for Qwen3-32B: 48/250 ran into the cap mid-thought (all parse failures)
# and the p90 thinking block among the rest was 3,589 tokens, i.e. the distribution was
# clipped at the cap. 2507-Thinking thinks longer still (Qwen recommends 32k), and with 3B
# active params tokens are cheap, so 4x the budget.
MAX_NEW_TOKENS = 16384 if ENABLE_THINKING else 1024

# Paths derive from MODEL_ID and the thinking mode so two configurations never share a
# checkpoint file -- run_generation_vllm() resumes from VLLM_JSONL, so a shared path would
# make a new run silently inherit the previous run's completions.
MODEL_SLUG = MODEL_ID.rstrip("/").split("/")[-1]
DATA_SLUG = os.path.splitext(os.path.basename(DATA_CSV))[0]
OUT_DIR = f"{REPO}/results/{DATA_SLUG}_{MODEL_SLUG}{'_think' if ENABLE_THINKING else ''}"

VLLM_JSONL = f"{OUT_DIR}/raw_generations_vllm.jsonl"   # this notebook writes here
RESULTS_CSV = f"{OUT_DIR}/results_vllm.csv"
SUMMARY_JSON = f"{OUT_DIR}/summary_vllm.json"   # headline metrics, machine-readable
SUMMARY_TXT = f"{OUT_DIR}/summary_vllm.txt"     # the printed report, verbatim
ALL_SUMMARIES_CSV = f"{REPO}/results/summary_all_vllm.csv"  # append-only log, one row per scoring run
# code_v1.ipynb's HF checkpoint. Read-only, and only for the optional comparison at the
# end -- this notebook never writes to it.
RAW_JSONL = f"{OUT_DIR}/raw_generations.jsonl"

os.makedirs(OUT_DIR, exist_ok=True)
print("writing to", OUT_DIR)
print(f"thinking={'on' if ENABLE_THINKING else 'off'}, max_new_tokens={MAX_NEW_TOKENS}")

df_eval = pd.read_csv(DATA_CSV)
print(df_eval.shape, df_eval.columns.tolist())
df_eval.head()

writing to /root/workspace/DFS/DeepRetro/results/uspto_50k_test_250_GLM-4.7-Flash
thinking=off, max_new_tokens=1024
(250, 4) ['input', 'output', 'reaction_type', 'cluster_id']


,input,output,reaction_type,cluster_id
0,CS(=O)c1cccc(-c2nc(C=O)ccc2OCCO[Si](C)(C)C(C)(...,CC(C)(C)[Si](C)(C)OCCOc1ccc(C=O)nc1Br.CS(=O)c1...,3,0
1,CC(C)=CCSc1ccc(Br)cc1,CC(C)=CCBr.Sc1ccc(Br)cc1,1,0
2,CCC1(c2ccc(C=O)s2)OCCO1,CCC1(c2cccs2)OCCO1.CN(C)C=O,3,0
3,O=C1Nc2ccccc2C1c1cc(Br)ccc1O,O=C1Nc2ccccc2C1(O)c1cc(Br)ccc1O,9,0
4,CCCCCCCCCCCCCCCCOCC(CN)CC#N,CCCCCCCCCCCCCCCCOCC(CC#N)CN=[N+]=[N-],9,0


In [9]:
# Tokenizer only -- no AutoModelForCausalLM. vLLM loads its own copy of the weights, so
# nothing heavyweight is pulled into this process. The tokenizer is still needed because
# build_prompt() renders the chat template.
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def build_prompt(molecule: str) -> str:
    """Render the chat prompt exactly as src/utils/llm.py::call_LLM builds its messages.

    Note: .replace() rather than .format() -- USER_PROMPT_OPENAI embeds a literal JSON
    schema with { } braces, so .format() would raise.

    enable_thinking is Qwen3's hard switch: False makes the template open the assistant
    turn with an empty <think>\\n\\n</think> so the model answers directly. (The soft
    switch, a trailing "/no_think" in the user message, does the same; not used here so
    the prompt text stays identical to the repo's.) For the 2507-Thinking checkpoints the
    kwarg is a no-op: their template always ends the generation prompt with "<think>\n".

    Base models ship no chat template, so fall back to a plain system+user concatenation
    rather than crashing.
    """
    user = USER_PROMPT_OPENAI.replace("{target_smiles}", molecule)
    if getattr(tokenizer, "chat_template", None):
        messages = [
            {"role": "system", "content": SYS_PROMPT_OPENAI},
            {"role": "user", "content": user},
        ]
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=ENABLE_THINKING,
        )
    return f"{SYS_PROMPT_OPENAI}\n\n{user}\n\n"


_p = build_prompt(df_eval.iloc[0]["input"])
print(_p[:1200])
print("...\n", repr(_p[-80:]))  # tail shows the empty <think> block (thinking off) or a bare "<think>\n" (2507-Thinking)

[gMASK]<sop><|system|>You are an expert organic chemist specializing in retrosynthesis. When given a target molecule, you will perform a single-step retrosynthesis, providing 3-5 possible precursor molecules or reactions that could lead to the formation of the target molecule. 

Present your final analysis in a specific JSON format. For each suggestion, provide the precursor molecules in SMILES notation and a brief explanation of the reaction type and any key conditions or reagents needed. Use standard organic chemistry notation and terminology in your explanations. 

If the molecule is too simple for meaningful retrosynthesis, state this in a single JSON object with an appropriate explanation.
<|user|>You are an expert organic chemist specializing in retrosynthesis. When given a target molecule, you will perform a single-step retrosynthesis, providing 3-5 possible precursor molecules or reactions that could lead to the formation of the target molecule. 

Present your final analysis in

In [10]:
# Ports of src/utils/llm.py::split_json_openAI and ::validate_split_json.
# Inlined rather than imported: src.utils.llm pulls in litellm/langfuse, which are not
# installed here. Two robustness additions over the originals, because a 7B model drops
# the <json> tags and emits real JSON far more often than Claude/o1 do:
#   - json.loads first, ast.literal_eval as fallback (the repo uses only the latter)
#   - regex fallback to a bare {...} block when the tags are missing

_BARE_JSON = re.compile(r"\{.*\"data\".*\}", re.DOTALL)
_THINK_END = "</think>"
 

def strip_thinking(res_text: str) -> str:
    """Drop the reasoning block so the parser only sees the answer.

    The model often rehearses a <json> block inside its thinking; find("<json>") would grab
    that draft instead of the final answer. Works whether or not the opening <think> is in
    the output (Qwen3-32B emits both tags; 2507-Thinking's template puts <think> in the
    prompt, so the output holds only </think>). Unchanged text if there is no </think>.
    """
    i = res_text.rfind(_THINK_END)
    return res_text[i + len(_THINK_END):] if i != -1 else res_text


def split_json_content(res_text: str):
    """Return (status, json_content). 200 on success, 502 on failure."""
    res_text = strip_thinking(res_text)
    start = res_text.find("<json>")
    end = res_text.find("</json>")
    if start != -1 and end != -1 and end > start:
        content = res_text[start + len("<json>"):end].strip()
        if content:
            return 200, content
    m = _BARE_JSON.search(res_text)  # tags missing -- salvage the object itself
    if m:
        return 200, m.group(0).strip()
    return 502, ""


def loads_lenient(json_content: str):
    try:
        return json.loads(json_content)
    except Exception:
        return ast.literal_eval(json_content)  # Python-literal style (repo behaviour)


def validate_split_json(json_content: str):
    """Return (status, molecules, explanations, confidence_scores). 504 on failure."""
    try:
        result = loads_lenient(json_content)
        return 200, result["data"], result["explanation"], result["confidence_scores"]
    except Exception:
        return 504, [], [], []


def parse_response(res_text: str):
    """Full response -> (status, proposals). proposals is a list of precursor-SMILES lists."""
    status, json_content = split_json_content(res_text)
    if status != 200:
        return status, []
    status, molecules, _expl, _conf = validate_split_json(json_content)
    if status != 200:
        return status, []
    # Normalise: a single flat list of strings means one proposal, not many.
    if molecules and all(isinstance(m, str) for m in molecules):
        molecules = [molecules]
    proposals = [
        [s for s in prop if isinstance(s, str) and s.strip()]
        for prop in molecules
        if isinstance(prop, list)
    ]
    proposals = [p for p in proposals if p]
    return (200, proposals) if proposals else (504, [])

In [11]:
def load_checkpoint(path):
    done = {}
    if os.path.exists(path):
        with open(path) as f:
            for line in f:
                line = line.strip()
                if line:
                    rec = json.loads(line)
                    done[rec["mol_no"]] = rec
    return done

In [12]:
def canon(smiles: str):
    try:
        return Chem.CanonSmiles(smiles)
    except Exception:
        return None


def max_frag(canon_list):
    """Largest fragment of a list of canonical SMILES, per Tetko et al. 2020 (MaxFrag).

    "Largest" is the reactant with the most heavy atoms; ties go to the longer canonical
    SMILES so the choice is deterministic. Returns None for an empty list.
    """
    mols = [(Chem.MolFromSmiles(s), s) for s in canon_list]
    mols = [(m, s) for m, s in mols if m is not None]
    if not mols:
        return None
    return max(mols, key=lambda ms: (ms[0].GetNumHeavyAtoms(), len(ms[1])))[1]


def score_proposal(pred_smiles_list, gt_canon):
    """Return (any_correct, all_correct, maxfrag_correct, not_common, missed, valid).

    maxfrag_correct follows Tetko et al. 2020: the largest predicted precursor must equal
    the largest ground-truth reactant. Unlike any/all_correct it does not require the
    precursor counts to match -- extra or missing small reagents do not hurt it.
    """
    pred = [canon(s) for s in pred_smiles_list]
    if any(p is None for p in pred):
        return 0, 0, 0, [s for s, p in zip(pred_smiles_list, pred) if p is None], gt_canon, False
    same_len = len(gt_canon) == len(pred)
    any_c = int(any(p in gt_canon for p in pred) and same_len)
    all_c = int(all(p in gt_canon for p in pred) and same_len)
    maxfrag_c = int(max_frag(pred) is not None and max_frag(pred) == max_frag(gt_canon))
    not_common = [p for p in pred if p not in gt_canon]
    missed = [g for g in gt_canon if g not in pred]
    return any_c, all_c, maxfrag_c, not_common, missed, True


def score_record(rec):
    """Score one generation record at top-1 and top-k."""
    gt_canon = [canon(s) for s in str(rec["output"]).split(".")]
    gt_canon = [g for g in gt_canon if g is not None]

    status, proposals = parse_response(rec["raw"])
    row = {
        "mol_no": rec["mol_no"],
        "input": rec["input"],
        "output": rec["output"],
        "parse_status": status,
        "n_proposals": len(proposals),
        "proposals": proposals,
        "any_correct": 0,
        "all_correct": 0,
        "maxfrag_correct": 0,
        "any_correct_topk": 0,
        "all_correct_topk": 0,
        "maxfrag_correct_topk": 0,
        "first_hit_rank": None,
        "not_common": [],
        "missed": gt_canon,
        "has_invalid_smiles": 0,
    }
    if status != 200 or not proposals:
        return row  # parse failure counts as incorrect, it is not dropped

    any_invalid = False
    for rank, prop in enumerate(proposals, start=1):
        any_c, all_c, maxfrag_c, not_common, missed, valid = score_proposal(prop, gt_canon)
        if not valid:
            any_invalid = True
        if rank == 1:  # top-1: exactly what prod_challenge scores
            row.update(any_correct=any_c, all_correct=all_c, maxfrag_correct=maxfrag_c,
                       not_common=not_common, missed=missed)
        row["any_correct_topk"] = max(row["any_correct_topk"], any_c)
        row["maxfrag_correct_topk"] = max(row["maxfrag_correct_topk"], maxfrag_c)
        if all_c and row["first_hit_rank"] is None:
            row["first_hit_rank"] = rank
        row["all_correct_topk"] = max(row["all_correct_topk"], all_c)
    row["has_invalid_smiles"] = int(any_invalid)
    return row

# Sanity check: the ground truth must score (1, 1, 1) against itself on every row.
_gt_ok = all(
    score_proposal([s for s in str(r["output"]).split(".")],
                   [canon(s) for s in str(r["output"]).split(".")])[:3] == (1, 1, 1)
    for _, r in df_eval.iterrows()
)
print("ground truth scores perfectly against itself:", _gt_ok)

ground truth scores perfectly against itself: True


In [13]:
# flashinfer JIT-compiles its sampling kernels on first use, and that build fails on this
# box (sm_120): curand.h is not on the include path, and forcing it there then hits a
# bundled-cccl header mismatch. We decode greedily, so those kernels buy us nothing --
# switch them off and use vLLM's PyTorch sampler. Must be set before vllm is imported.
os.environ.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")

from vllm import LLM, SamplingParams

# Sized from what is actually free, so this works even if another kernel still holds GPU
# memory. gpu_memory_utilization is a fraction of *total*, not of free.
_free, _total = torch.cuda.mem_get_info()
GPU_UTIL = round(min(0.85, (_free / _total) * 0.90), 2)
print(f"free {_free/1e9:.0f}GB / {_total/1e9:.0f}GB -> gpu_memory_utilization={GPU_UTIL}")

# Context = ~600-token prompt + generation budget, with headroom. Follows MAX_NEW_TOKENS so
# a thinking run (16384 new) is not cut off by the engine before the sampler's own cap.
MAX_MODEL_LEN = MAX_NEW_TOKENS + 1024

# Qwen3-30B-A3B in bf16 is ~61GB of weights; at GPU_UTIL=0.85 on the 96GB card that leaves
# ~20GB of KV cache. Per token this model needs ~96KB of KV (48 layers x 4 KV heads x 128
# dim x K+V x bf16), so ~200k tokens fit: a dozen sequences at the full 17k context, many
# more at typical lengths. vLLM schedules within that, so 250 prompts is fine.
llm = LLM(
    model=MODEL_ID,
    dtype="bfloat16",
    max_model_len=MAX_MODEL_LEN,
    gpu_memory_utilization=GPU_UTIL,
    enable_prefix_caching=True,   # the ~600-token shared preamble, computed once
)

free 101GB / 102GB -> gpu_memory_utilization=0.85
INFO 09-21 19:20:32 [api_utils.py:273] non-default args: {'dtype': 'bfloat16', 'max_model_len': 2048, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': 'zai-org/GLM-4.7-Flash'}
WARNING 09-21 19:20:32 [envs.py:2041] Unknown vLLM environment variable detected: VLLM_BUILD_PIPELINE
WARNING 09-21 19:20:32 [envs.py:2041] Unknown vLLM environment variable detected: VLLM_TEST_ENDPOINT
WARNING 09-21 19:20:32 [envs.py:2041] Unknown vLLM environment variable detected: VLLM_BUILD_COMMIT
WARNING 09-21 19:20:32 [envs.py:2041] Unknown vLLM environment variable detected: VLLM_BUILD_URL
WARNING 09-21 19:20:32 [envs.py:2041] Unknown vLLM environment variable detected: VLLM_IMAGE_TAG
INFO 09-21 19:20:39 [model.py:619] Resolved architecture: Glm4MoeLiteForCausalLM
INFO 09-21 19:20:39 [model.py:1776] Using max model len 2048
INFO 09-21 19:20:39 [scheduler.py:252] Chunked prefill is enabled with max_num_batche

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

WARNING 09-21 19:20:42 [system_utils.py:157] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(EngineCore pid=63192) INFO 09-21 19:20:47 [core.py:114] Initializing a V1 LLM engine (v0.25.1) with config: model='zai-org/GLM-4.7-Flash', speculative_config=None, tokenizer='zai-org/GLM-4.7-Flash', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_

Loading safetensors checkpoint shards:   0% Completed | 0/48 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:   4% Completed | 2/48 [00:00<00:05,  7.73it/s]
Loading safetensors checkpoint shards:   6% Completed | 3/48 [00:00<00:06,  6.99it/s]
Loading safetensors checkpoint shards:   8% Completed | 4/48 [00:00<00:06,  6.71it/s]
Loading safetensors checkpoint shards:  10% Completed | 5/48 [00:00<00:06,  6.50it/s]
Loading safetensors checkpoint shards:  12% Completed | 6/48 [00:00<00:06,  6.42it/s]
Loading safetensors checkpoint shards:  15% Completed | 7/48 [00:01<00:06,  6.31it/s]
Loading safetensors checkpoint shards:  17% Completed | 8/48 [00:01<00:06,  6.30it/s]
Loading safetensors checkpoint shards:  19% Completed | 9/48 [00:01<00:06,  6.26it/s]
Loading safetensors checkpoint shards:  21% Completed | 10/48 [00:01<00:06,  6.27it/s]
Loading safetensors checkpoint shards:  23% Completed | 11/48 [00:01<00:05,  6.24it/s]
Loading safetensors checkpoint shards:  25% Completed | 12/4

(EngineCore pid=63192) INFO 09-21 19:24:32 [default_loader.py:430] Loading weights took 7.67 seconds
(EngineCore pid=63192) INFO 09-21 19:24:32 [unquantized.py:334] Using MoEPrepareAndFinalizeNoDPEPModular
(EngineCore pid=63192) INFO 09-21 19:24:33 [gpu_model_runner.py:5306] Model loading took 55.87 GiB memory and 223.332783 seconds
(EngineCore pid=63192) INFO 09-21 19:24:41 [backends.py:1089] Using cache directory: /root/.cache/vllm/torch_compile_cache/a5d16d6ade/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=63192) INFO 09-21 19:24:41 [backends.py:1148] Dynamo bytecode transform time: 7.78 s
(EngineCore pid=63192) INFO 09-21 19:24:47 [backends.py:378] Cache the graph of compile range (1, 16384) for later use


...

(EngineCore pid=63192) INFO 09-21 19:24:54 [backends.py:393] Compiling a graph for compile range (1, 16384) takes 13.05 s
(EngineCore pid=63192) INFO 09-21 19:24:57 [decorators.py:708] saved AOT compiled function to /root/.cache/vllm/torch_compile_cache/torch_aot_compile/662af2d4773e3521ab254d6bf8f75700574ffd373c82830929e1ade5988b0666/rank_0_0/model
(EngineCore pid=63192) INFO 09-21 19:24:57 [monitor.py:53] torch.compile took 24.21 s in total
(EngineCore pid=63192) INFO 09-21 19:25:01 [monitor.py:81] Initial profiling/warmup run took 3.47 s
(EngineCore pid=63192) INFO 09-21 19:25:05 [gpu_model_runner.py:6534] Profiling CUDA graph memory: PIECEWISE=51 (largest=512), FULL=51 (largest=512)
(EngineCore pid=63192) INFO 09-21 19:25:08 [gpu_model_runner.py:6639] Estimated CUDA graph memory: 0.77 GiB total
(EngineCore pid=63192) INFO 09-21 19:25:09 [gpu_worker.py:538] Available KV cache memory: 20.45 GiB
(EngineCore pid=63192) INFO 09-21 19:25:09 [gpu_worker.py:553] CUDA graph memory profiling

(EngineCore pid=63192) 2026-09-21 19:25:09,672 - INFO - autotuner.py:651 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
[AutoTuner]: Tuning trtllm::fused_moe::gemm1:   0%|          | 0/21 [00:00<?, ?profile/s]2026-09-21 19:25:09,740 - INFO - autotuner.py:1351 - flashinfer.jit: [Autotuner]: Skipped 2 unsupported tactic(s) for trtllm::fused_moe::gemm1 (enable debug logs to see details)
(EngineCore pid=63192) 2026-09-21 19:25:09,785 - INFO - autotuner.py:1351 - flashinfer.jit: [Autotuner]: Skipped 2 unsupported tactic(s) for trtllm::fused_moe::gemm1 (enable debug logs to see details)
(EngineCore pid=63192) 2026-09-21 19:25:09,833 - INFO - autotuner.py:1351 - flashinfer.jit: [Autotuner]: Skipped 2 unsupported tactic(s) for trtllm::fused_moe::gemm1 (enable debug logs to see details)
[AutoTuner]: Tuning trtllm::fused_moe::gemm1:  14%|█▍        | 3/21 [00:00<00:00, 20.67profile/s]2026-09-21 19:25:09,888 - INFO - autotuner.py:1351 - flashinfer.jit: [Autotuner]: Skipped 2 unsuppor

(EngineCore pid=63192) INFO 09-21 19:25:14 [kernel_warmup.py:209] FlashInfer autotune cache loaded on rank 0 from /root/.cache/vllm/flashinfer_autotune_cache/0.6.13/120f/ded417dcab6177c4d391829b45abdefdc7a8913654fb8dee08c7df0765fd21fa/autotune_configs.json.
(EngineCore pid=63192) INFO 09-21 19:25:14 [cutedsl_warmup.py:97] Skipping CuTeDSL warmup because no compile units were requested.


Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:11<00:00,  4.57it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 51/51 [00:06<00:00,  7.33it/s]


(EngineCore pid=63192) INFO 09-21 19:25:32 [gpu_model_runner.py:6707] Graph capturing finished in 19 secs, took 0.95 GiB
(EngineCore pid=63192) INFO 09-21 19:25:32 [gpu_worker.py:771] CUDA graph pool memory: 0.95 GiB (actual), 0.77 GiB (estimated), difference: 0.18 GiB (18.5%).
(EngineCore pid=63192) INFO 09-21 19:25:33 [jit_monitor.py:73] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.
(EngineCore pid=63192) INFO 09-21 19:25:33 [core.py:337] init engine (profile, create kv cache, warmup model) took 59.93 s (compilation: 24.21 s)
(EngineCore pid=63192) INFO 09-21 19:25:34 [vllm.py:1042] Asynchronous scheduling is enabled.
(EngineCore pid=63192) INFO 09-21 19:25:34 [kernel.py:292] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])


In [ ]:
# Non-thinking: greedy, same token budget as the HF loop this replaces, so the two are
# comparable. Thinking: Qwen's model cards say greedy decoding in thinking mode causes
# endless repetition; temperature=0.6, top_p=0.95, top_k=20 is their recommendation for
# both Qwen3-32B and Qwen3-30B-A3B-Thinking-2507. Either way, stop as soon
# as the object closes rather than rambling on to the cap; parse_response() looks for the
# closing tag, so it is kept in the output.
if ENABLE_THINKING:
    _decode = dict(temperature=0.6, top_p=0.95, top_k=20)
else:
    _decode = dict(temperature=1.0)  # greedy, matching the repo's temperature=0.0

SAMPLING = SamplingParams(
    **_decode,
    max_tokens=MAX_NEW_TOKENS,
    stop=["</json>"],
    include_stop_str_in_output=True,
)


def run_generation_vllm(df, limit=None):
    """Generate completions for df, appending to VLLM_JSONL and skipping finished rows.

    No BATCH_SIZE: vLLM schedules continuously, so one call covers the whole set and a
    finished sequence leaves the batch instead of waiting for its slowest neighbour.
    """
    done = load_checkpoint(VLLM_JSONL)
    todo = [(i, row) for i, row in df.iterrows() if int(i) not in done]
    if limit is not None:
        todo = todo[:limit]
    print(f"{len(done)} already done, generating {len(todo)}")
    if not todo:
        return

    prompts = [build_prompt(row["input"]) for _, row in todo]
    t0 = time.time()
    outs = llm.generate(prompts, SAMPLING)
    elapsed = time.time() - t0

    # One write at the end rather than per batch -- acceptable when the whole run is
    # minutes rather than half an hour.
    with open(VLLM_JSONL, "a") as fout:
        for (i, row), out in zip(todo, outs):
            fout.write(json.dumps({
                "mol_no": int(i),
                "model_id": MODEL_ID,
                "thinking": ENABLE_THINKING,
                "input": row["input"],
                "output": row["output"],
                "raw": out.outputs[0].text,
            }) + "\n")
        fout.flush()

    gen_toks = sum(len(o.outputs[0].token_ids) for o in outs)
    truncated = sum(o.outputs[0].finish_reason == "length" for o in outs)
    # Thinking that never closed is a guaranteed parse failure -- surface it here so a cap
    # problem is visible before scoring.
    unclosed = sum("</think>" not in o.outputs[0].text for o in outs) if ENABLE_THINKING else 0
    print(f"{len(todo)} molecules in {elapsed/60:.2f} min "
          f"({elapsed/len(todo):.2f} s/mol, {gen_toks/elapsed:.0f} tok/s), "
          f"{truncated} hit the {MAX_NEW_TOKENS}-token cap, "
          f"{unclosed} never closed </think>")

In [15]:
# Smoke test: one molecule end to end, through the same parser the scoring uses.
_smoke = llm.generate([build_prompt(df_eval.iloc[0]["input"])], SAMPLING)[0].outputs[0].text
print(_smoke[:2000])
print("\n--- parsed ---")
print(parse_response(_smoke))

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1/1 [00:12<00:00, 12.68s/it, est. speed input: 40.78 toks/s, output: 49.14 toks/s]

<json>
{
  "data": [
    [
      "OCCO[Si](C)(C)C(C)(C)C",
      "O=C(N)c1ccc(cc1)C(=O)Nc2ccc(cc2)C(=O)Nc3ccc(cc3)C(=O)N"
    ],
    [
      "OCCO[Si](C)(C)C(C)(C)C",
      "O=C(Nc1ccc(cc1)C(=O)Nc2ccc(cc2)C(=O)N)c3ccc(cc3)C(=O)N"
    ],
    [
      "OCCO[Si](C)(C)C(C)(C)C",
      "O=C(Nc1ccc(cc1)C(=O)N)c2ccc(cc2)C(=O)Nc3ccc(cc3)C(=O)N"
    ],
    [
      "OCCO[Si](C)(C)C(C)(C)C",
      "O=C(Nc1ccc(cc1)C(=O)N)c2ccc(cc2)C(=O)Nc3ccc(cc3)C(=O)N"
    ]
  ],
  "explanation": [
    "This is a convergent synthesis via a double amide coupling (e.g., HATU, EDCI, or DCC) between a silyl-protected diol (the silyl ether) and a tripeptide-like fragment containing three aniline-derived amide linkages. The target is formed by linking the central phenyl ring to the silyl-protected diol via an amide bond.",
    "This is a convergent synthesis via a double amide coupling (e.g., HATU, EDCI, or DCC) between a silyl-protected diol and a dipeptide-like fragment. The target is formed by linking the central ph

In [16]:
# Dry run on the first 10. Re-running this cell should say "10 already done, generating 0"
# -- that confirms the checkpoint works before committing to the full set.
run_generation_vllm(df_eval, limit=10)

0 already done, generating 10


Rendering prompts:   0%|          | 0/10 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 10/10 [00:30<00:00,  3.03s/it, est. speed input: 165.82 toks/s, output: 229.14 toks/s]

10 molecules in 0.50 min (3.03 s/mol, 229 tok/s), 2 hit the 1024-token cap, 0 never closed </think>


In [17]:
# Full run over all 250. Non-thinking: ~1 min, against ~34 min for the HF loop. Thinking at
# 16k tokens: expect ~10 min. Resumable, so it is safe to interrupt and re-run.
run_generation_vllm(df_eval)

10 already done, generating 240


Rendering prompts:   0%|          | 0/240 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 240/240 [00:57<00:00,  4.21it/s, est. speed input: 2158.88 toks/s, output: 3211.25 toks/s]

240 molecules in 0.95 min (0.24 s/mol, 3199 tok/s), 40 hit the 1024-token cap, 0 never closed </think>


In [18]:
# Score every vLLM row, then -- if code_v1.ipynb's HF checkpoint happens to be present --
# put the two engines side by side over the identical set of molecules. That comparison is
# the regression check: a large gap means a prompt or stop-token problem, not a real change
# in model accuracy. The notebook stands alone without it.
recs_vllm = load_checkpoint(VLLM_JSONL)
df_vllm = pd.DataFrame([score_record(r) for r in recs_vllm.values()]).set_index("mol_no")
df_vllm = df_vllm.sort_index()
df_vllm.to_csv(RESULTS_CSV)

n = len(df_vllm)

# One dict is the source of truth; the printout, the JSON and the cross-model CSV are all
# rendered from it so they cannot drift apart. Percentages are derived from the raw counts.
def _counts_and_pcts(all_c, any_c, maxfrag_c):
    counts = {"n_all_correct": int(all_c.sum()), "n_any_correct": int(any_c.sum()),
              "n_maxfrag": int(maxfrag_c.sum())}
    pcts = {k[2:]: 100 * v / n for k, v in counts.items()}
    return {**pcts, **counts}


summary = {
    "run": os.path.basename(OUT_DIR),
    "model_id": MODEL_ID,
    "thinking": ENABLE_THINKING,
    "max_new_tokens": MAX_NEW_TOKENS,
    "data_csv": DATA_CSV,
    "n_scored": int(n),
    "n_total": int(len(df_eval)),
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
    "top1": _counts_and_pcts(df_vllm.all_correct, df_vllm.any_correct, df_vllm.maxfrag_correct),
    "topk": _counts_and_pcts(df_vllm.all_correct_topk, df_vllm.any_correct_topk,
                             df_vllm.maxfrag_correct_topk),
    "parse_failures_pct": 100 * (df_vllm.parse_status != 200).sum() / n,
    "invalid_smiles_pct": 100 * df_vllm.has_invalid_smiles.sum() / n,
    "mean_proposals": df_vllm.n_proposals.mean(),
}

report = "\n".join([
    f"model: {MODEL_ID}",
    f"scored {n} / {len(df_eval)} molecules from {VLLM_JSONL}",
    "",
    "=== top-1 (first proposal -- prod_challenge protocol) ===",
    f"All correct % {summary['top1']['all_correct']}  ({summary['top1']['n_all_correct']}/{n})",
    f"Any correct % {summary['top1']['any_correct']}  ({summary['top1']['n_any_correct']}/{n})",
    f"MaxFrag     % {summary['top1']['maxfrag']}  ({summary['top1']['n_maxfrag']}/{n})",
    "",
    "=== top-k (best of the 3-5 proposals) ===",
    f"All correct % {summary['topk']['all_correct']}  ({summary['topk']['n_all_correct']}/{n})",
    f"Any correct % {summary['topk']['any_correct']}  ({summary['topk']['n_any_correct']}/{n})",
    f"MaxFrag     % {summary['topk']['maxfrag']}  ({summary['topk']['n_maxfrag']}/{n})",
    "",
    "=== response quality ===",
    f"parse failures  : {summary['parse_failures_pct']:.2f}%",
    f"invalid SMILES  : {summary['invalid_smiles_pct']:.2f}%",
    f"mean proposals  : {summary['mean_proposals']:.2f}",
    "",
    f"Denominator is all {n} scored rows -- parse failures count as incorrect.",
])
print(report)

with open(SUMMARY_TXT, "w") as f:
    f.write(report + "\n")
with open(SUMMARY_JSON, "w") as f:
    json.dump(summary, f, indent=2)

# Cross-model log: append one flat row per scoring run. Re-scoring adds another row rather
# than replacing the old one; the timestamp column tells them apart.
_flat = {k: v for k, v in summary.items() if not isinstance(v, dict)}
_flat.update({f"top1_{k}": v for k, v in summary["top1"].items()})
_flat.update({f"topk_{k}": v for k, v in summary["topk"].items()})
_row = pd.DataFrame([_flat])
if os.path.exists(ALL_SUMMARIES_CSV):
    _prev = pd.read_csv(ALL_SUMMARIES_CSV)
    if list(_prev.columns) == list(_row.columns):
        _row.to_csv(ALL_SUMMARIES_CSV, mode="a", index=False, header=False)
    else:  # column set changed (new metric added): rewrite so the header stays aligned
        pd.concat([_prev, _row], ignore_index=True).to_csv(ALL_SUMMARIES_CSV, index=False)
else:
    _row.to_csv(ALL_SUMMARIES_CSV, index=False)
print(f"\nwrote {SUMMARY_TXT}\n      {SUMMARY_JSON}\n      appended to {ALL_SUMMARIES_CSV}")


def _summary(d):
    return [100 * d.all_correct.mean(), 100 * d.any_correct.mean(),
            100 * d.maxfrag_correct.mean(),
            100 * d.all_correct_topk.mean(), 100 * d.any_correct_topk.mean(),
            100 * d.maxfrag_correct_topk.mean(),
            100 * (d.parse_status != 200).mean(), d.n_proposals.mean()]


comparison = None
if os.path.exists(RAW_JSONL):
    recs_hf = load_checkpoint(RAW_JSONL)
    common = sorted(set(recs_vllm) & set(recs_hf))
    if common:
        df_v = pd.DataFrame([score_record(recs_vllm[i]) for i in common]).set_index("mol_no")
        df_h = pd.DataFrame([score_record(recs_hf[i]) for i in common]).set_index("mol_no")
        comparison = pd.DataFrame(
            {"vLLM": _summary(df_v), "HF (batch=4)": _summary(df_h)},
            index=["top-1 all_correct %", "top-1 any_correct %", "top-1 maxfrag %",
                   "top-k all_correct %", "top-k any_correct %", "top-k maxfrag %",
                   "parse failures %", "mean proposals"],
        ).round(2)
        print(f"\ncomparing {len(common)} molecules present in both runs")
else:
    print(f"\nno HF run at {RAW_JSONL} -- skipping engine comparison")

comparison if comparison is not None else df_vllm.head(10)

model: zai-org/GLM-4.7-Flash
scored 250 / 250 molecules from /root/workspace/DFS/DeepRetro/results/uspto_50k_test_250_GLM-4.7-Flash/raw_generations_vllm.jsonl

=== top-1 (first proposal -- prod_challenge protocol) ===
All correct % 0.0  (0/250)
Any correct % 2.0  (5/250)
MaxFrag     % 1.2  (3/250)

=== top-k (best of the 3-5 proposals) ===
All correct % 0.4  (1/250)
Any correct % 2.4  (6/250)
MaxFrag     % 1.2  (3/250)

=== response quality ===
parse failures  : 16.40%
invalid SMILES  : 16.80%
mean proposals  : 3.52

Denominator is all 250 scored rows -- parse failures count as incorrect.

wrote /root/workspace/DFS/DeepRetro/results/uspto_50k_test_250_GLM-4.7-Flash/summary_vllm.txt
      /root/workspace/DFS/DeepRetro/results/uspto_50k_test_250_GLM-4.7-Flash/summary_vllm.json
      appended to /root/workspace/DFS/DeepRetro/results/summary_all_vllm.csv

no HF run at /root/workspace/DFS/DeepRetro/results/uspto_50k_test_250_GLM-4.7-Flash/raw_generations.jsonl -- skipping engine comparison


<unknown>:4: SyntaxWarning: invalid escape sequence '\C'
<unknown>:8: SyntaxWarning: invalid escape sequence '\C'
<unknown>:12: SyntaxWarning: invalid escape sequence '\C'
<unknown>:16: SyntaxWarning: invalid escape sequence '\C'
<unknown>:20: SyntaxWarning: invalid escape sequence '\C'


,input,output,parse_status,n_proposals,proposals,any_correct,all_correct,maxfrag_correct,any_correct_topk,all_correct_topk,maxfrag_correct_topk,first_hit_rank,not_common,missed,has_invalid_smiles
mol_no,,,,,,,,,,,,,,,
0,CS(=O)c1cccc(-c2nc(C=O)ccc2OCCO[Si](C)(C)C(C)(...,CC(C)(C)[Si](C)(C)OCCOc1ccc(C=O)nc1Br.CS(=O)c1...,502,0,[],0,0,0,0,0,0,NaN,[],"[CC(C)(C)[Si](C)(C)OCCOc1ccc(C=O)nc1Br, CS(=O)...",0
1,CC(C)=CCSc1ccc(Br)cc1,CC(C)=CCBr.Sc1ccc(Br)cc1,200,4,"[[C=CCSc1ccc(Br)cc1, C=CCS], [C=CCS, C=CCBr], ...",0,0,0,0,0,0,NaN,"[C=CCSc1ccc(Br)cc1, C=CCS]","[CC(C)=CCBr, Sc1ccc(Br)cc1]",0
2,CCC1(c2ccc(C=O)s2)OCCO1,CCC1(c2cccs2)OCCO1.CN(C)C=O,200,5,"[[CCOC1=CC=C(C=C1)C(=O)O, OCCO, C1=CC=C(C=C1)C...",0,0,0,0,0,0,NaN,"[CCOc1ccc(C(=O)O)cc1, OCCO, O=C(O)c1ccccc1]","[CCC1(c2cccs2)OCCO1, CN(C)C=O]",0
3,O=C1Nc2ccccc2C1c1cc(Br)ccc1O,O=C1Nc2ccccc2C1(O)c1cc(Br)ccc1O,200,4,"[[O=C1Nc2ccccc2C1c1cc(Br)ccc1O, O=C1Nc2ccccc2C...",0,0,0,0,0,0,NaN,"[O=C1Nc2ccccc2C1c1cc(Br)ccc1O, O=C1Nc2ccccc2C1...",[O=C1Nc2ccccc2C1(O)c1cc(Br)ccc1O],1
4,CCCCCCCCCCCCCCCCOCC(CN)CC#N,CCCCCCCCCCCCCCCCOCC(CC#N)CN=[N+]=[N-],200,5,"[[CCCCCCCCCCCCCCCCOCC(CN)C#N, CCCCCCCCCCCCCCCC...",0,0,0,0,0,0,NaN,"[CCCCCCCCCCCCCCCCOCC(C#N)CN, CCCCCCCCCCCCCCCCO...",[CCCCCCCCCCCCCCCCOCC(CC#N)CN=[N+]=[N-]],0
5,CCCn1cc(C=O)nc1C,CCCI.Cc1nc(C=O)c[nH]1,502,0,[],0,0,0,0,0,0,NaN,[],"[CCCI, Cc1nc(C=O)c[nH]1]",0
6,Cc1cccc(N2CCN(CCCc3cc(-c4ccccc4)n(C(C)(C)C)n3)...,CC(C)(C)n1nc(CCC=O)cc1-c1ccccc1.Cc1cccc(N2CCNC...,200,4,[[Cc1cccc(N2CCN(CCCc3cc(-c4ccccc4)n(C(C)(C)C)n...,0,0,0,0,0,0,NaN,[Cc1cccc(N2CCN(CCCc3cc(-c4ccccc4)n(C(C)(C)C)n3...,"[CC(C)(C)n1nc(CCC=O)cc1-c1ccccc1, Cc1cccc(N2CC...",0
7,COc1ccccc1C1CCCCC1CO,CCOC(=O)C1CCCCC1c1ccccc1OC,200,4,"[[COc1ccccc1C1CCCCC1O, Oc1ccccc1C1CCCCC1C], [C...",0,0,0,0,0,0,NaN,"[COc1ccccc1C1CCCCC1O, CC1CCCCC1c1ccccc1O]",[CCOC(=O)C1CCCCC1c1ccccc1OC],0
8,CC1(c2cncc(C=O)c2)OCCO1,CC1(c2cncc(Br)c2)OCCO1.CN(C)C=O,200,4,"[[CC1(c2cncc(C=O)c2)OCCO1, OCCO], [CC1(c2cncc(...",0,0,0,0,0,0,NaN,"[CC1(c2cncc(C=O)c2)OCCO1, OCCO]","[CC1(c2cncc(Br)c2)OCCO1, CN(C)C=O]",0
